For this project, I'm using the United States Utility Rate Database which provides rate structure information for over 3,700 U.S. utilities. 

In [15]:
import pandas as pd

df = pd.read_csv("usurdb.csv")

/var/folders/xm/dgtcr8j1797dx3lb2x3q5rmc0000gn/T/ipykernel_36877/1888131420.py:3: DtypeWarning: Columns (3,10,29,375,405,410,415,420,425,430,450,455,460,465,470,485,490,495,505,510,515,520,530,535,540,545,555,560,565,570,575,580,585,590,595,600,605,610,615,620,625,630,635,640,645,650,655,660,665,670,675,680,685,690,695,700,705,710,715,720,723,736) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("usurdb.csv")


Understanding the data better:

In [16]:
print(len(df))
df.dtypes

58920


label                                         object
eiaid                                        float64
name                                          object
is_default                                    object
startdate                                     object
                                              ...   
coincidentratestructure/period2/tier0max     float64
coincidentratestructure/period2/tier0rate    float64
coincidentratestructure/period2/tier0adj     float64
coincidentratestructure/period2/tier0unit    float64
coincidentschedule                            object
Length: 737, dtype: object

Since this is a large, messy dataset, I want to identify the columns that have information about rate structures specifically. 

In [17]:
num = df.select_dtypes(include="number")
coverage = num.notna().sum().sort_values(ascending=False)
print(coverage.head(20))


eiaid                                    58920
fixedchargefirstmeter                    53714
energyratestructure/period0/tier0rate    52085
energyratestructure/period0/tier0adj     25321
energyratestructure/period1/tier0rate    21592
flatdemandstructure/period0/tier0rate    21363
flatDemandMonth_apr                      21250
flatDemandMonth_may                      21250
flatDemandMonth_jun                      21250
flatDemandMonth_jul                      21250
flatDemandMonth_aug                      21250
flatDemandMonth_mar                      21250
flatDemandMonth_oct                      21250
flatDemandMonth_nov                      21250
flatDemandMonth_dec                      21250
flatDemandMonth_jan                      21250
flatDemandMonth_sep                      21250
flatDemandMonth_feb                      21250
energyratestructure/period0/tier0max     14970
energyratestructure/period0/tier1rate    14949
dtype: int64


For the specific variable I would like to analyze, which is the first-tier energy rate in the first time-of-use period of a utility rate plan, usually the price per kWh.

Since some values are not in price per Kilo-Watt hours, which is the standard unit, we can drop those values.

In [22]:
rate = "energyratestructure/period0/tier0rate"
rate = "energyratestructure/period0/tier0rate"
unit = "energyratestructure/period0/tier0unit"

kwh = df[df[unit] == "kWh"]
values = kwh[rate].dropna()

print(len(values), values.mean(), values.median(), values.mode()[0])


41278 0.09418624827555114 0.073 0.0


Across 41,278 utility rates measured in kWh, the mean first-tier energy rate is about $0.094 per kWh, the median is $0.073, and the most common value (mode) is $0.00, this likely reflects entries with no charge or unfilled values.


In [23]:
print(df[rate].mean(), df[rate].median(), df[rate].mode()[0])

0.4361511736658961 0.0727 0.0


In [21]:
df["energyratestructure/period0/tier0unit"].value_counts()

energyratestructure/period0/tier0unit
kWh             41642
kWh/kW           2230
kWh daily         872
kWh/kVA           126
kWh/hp            118
kWh/kW daily        2
Name: count, dtype: int64